# Submission Ready Colab Runner

This notebook is designed for the full refresh workflow:

1. Open this notebook in Google Colab.
2. Upload `submission_ready.zip` to the Colab file area, or let the first cell prompt you.
3. Click `Runtime -> Run all`.

It will then:

- extract `submission_ready.zip`
- install dependencies from `requirements.txt`
- rerun baseline calibration and save baseline shock income snapshots
- rerun the PPO sweep used for PPO shock income snapshots
- rerun the multi-seed experiments
- rerun statistical analysis and regenerate the four figures
- package the refreshed folder and trigger a download

Use a GPU runtime if Colab gives you one. The scripts use PyTorch's automatic device selection.


In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

UPLOAD_DIR = Path('/content')
TARGET_DIR = UPLOAD_DIR / 'submission_ready'

existing_zip = None
for candidate in [UPLOAD_DIR / 'submission_ready.zip'] + sorted(UPLOAD_DIR.glob('*.zip')):
    if candidate.exists():
        existing_zip = candidate
        break

if existing_zip is None:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise ValueError('No zip file uploaded. Please upload submission_ready.zip.')
    zip_path = UPLOAD_DIR / zip_names[0]
else:
    zip_path = existing_zip

if TARGET_DIR.exists():
    shutil.rmtree(TARGET_DIR)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(UPLOAD_DIR)

if not TARGET_DIR.exists():
    candidates = [p for p in UPLOAD_DIR.iterdir() if p.is_dir() and (p / 'final_submission_en.ipynb').exists()]
    if len(candidates) == 1:
        TARGET_DIR = candidates[0]
    else:
        raise FileNotFoundError('Could not find extracted submission_ready folder.')

os.chdir(TARGET_DIR)
TARGET_DIR

In [ ]:
import contextlib
import csv
import importlib.util
import io
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/content/submission_ready')
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
DOCS_DIR = PROJECT_ROOT / 'docs'

TOTAL_TIMESTEPS = 20000
PRIMARY_ALPHA = 0.2
PRIMARY_SEED = 0

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

status_handle = display(HTML('<b>Runner ready.</b>'), display_id=True)

def set_status(text: str) -> None:
    status_handle.update(HTML(f"<div style='font-size:16px; font-weight:600'>{text}</div>"))

def load_module(name: str, relative_path: str):
    module_path = PROJECT_ROOT / relative_path
    spec = importlib.util.spec_from_file_location(name, module_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

set_status('Notebook extracted and helper utilities loaded.')


In [ ]:
set_status('Installing dependencies...')
setup_bar = tqdm(total=3, desc='Environment setup', unit='step')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
setup_bar.update(1)

import matplotlib
import torch

setup_bar.update(1)
setup_bar.update(1)
setup_bar.close()

set_status(f"Dependencies installed. CUDA available: {torch.cuda.is_available()}")


In [ ]:
set_status('Running baseline calibration, PPO snapshots, and multi-seed experiments...')

experiment_bar = tqdm(total=3, desc='Experiment pipeline', unit='step')

subprocess.run([sys.executable, 'scripts/evaluate_baselines.py'], check=True)
experiment_bar.update(1)

subprocess.run([
    sys.executable,
    'scripts/train_ppo.py',
    '--seed', str(PRIMARY_SEED),
    '--total-timesteps', str(TOTAL_TIMESTEPS),
], check=True)
experiment_bar.update(1)

subprocess.run([
    sys.executable,
    'scripts/run_multi_seed_experiments.py',
    '--total-timesteps', str(TOTAL_TIMESTEPS),
], check=True)
experiment_bar.update(1)

experiment_bar.close()
set_status('Experiment CSVs and income snapshots were refreshed.')


In [ ]:
set_status('Running statistical analysis and regenerating figures...')

subprocess.run([
    sys.executable,
    'scripts/statistical_analysis.py',
    '--alpha', str(PRIMARY_ALPHA),
], check=True)

summary_df = pd.read_csv(RESULTS_DIR / 'statistical_summary.csv')
significance_df = pd.read_csv(RESULTS_DIR / 'significance_tests.csv')

set_status(
    'Statistical analysis finished. '
    f'Summary rows: {len(summary_df)}; significance rows: {len(significance_df)}.'
)


In [ ]:
set_status('Generating final English figures...')

import matplotlib.pyplot as plt
import numpy as np

english_bar = tqdm(total=4, desc='English figure pipeline', unit='figure')
summary_df = pd.read_csv(RESULTS_DIR / 'statistical_summary.csv')
baseline_income_df = pd.read_csv(RESULTS_DIR / 'baseline_shock_incomes.csv')
ppo_income_df = pd.read_csv(RESULTS_DIR / 'ppo_shock_incomes.csv')
ppo_income_df = ppo_income_df[
    (ppo_income_df['scene'] == 'shock')
    & (ppo_income_df['alpha'].astype(float).round(1) == PRIMARY_ALPHA)
]

plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

primary_df = summary_df[(summary_df['algorithm'] != 'PPO') | (summary_df['alpha'].fillna(0).round(1) == PRIMARY_ALPHA)].copy()
algorithms = ['Local-First', 'PPO', 'Demand-Greedy']
positions = np.arange(len(algorithms))

def ordered_rows(metric_prefix: str, scene: str):
    means = []
    errors = []
    for algo in algorithms:
        row = primary_df[(primary_df['algorithm'] == algo) & (primary_df['scene'] == scene)]
        means.append(row[f'{metric_prefix}_mean'].iloc[0])
        errors.append(row[f'{metric_prefix}_sem'].iloc[0])
    return means, errors

plt.figure(figsize=(8, 4))
normal_means, normal_errs = ordered_rows('revenue', 'normal')
shock_means, shock_errs = ordered_rows('revenue', 'shock')
plt.bar(positions - 0.2, normal_means, width=0.4, yerr=normal_errs, capsize=5, label='normal')
plt.bar(positions + 0.2, shock_means, width=0.4, yerr=shock_errs, capsize=5, label='shock')
plt.xticks(positions, algorithms)
plt.ylabel('Episode revenue')
plt.title('Episode revenue by scene')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'episode_revenue_by_scene.png', dpi=150)
plt.close()
english_bar.update(1)

plt.figure(figsize=(8, 4))
normal_means, normal_errs = ordered_rows('gini', 'normal')
shock_means, shock_errs = ordered_rows('gini', 'shock')
plt.bar(positions - 0.2, normal_means, width=0.4, yerr=normal_errs, capsize=5, label='normal')
plt.bar(positions + 0.2, shock_means, width=0.4, yerr=shock_errs, capsize=5, label='shock')
plt.xticks(positions, algorithms)
plt.ylabel('Final episode Gini')
plt.title('Final episode Gini by scene')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_gini_by_scene.png', dpi=150)
plt.close()
english_bar.update(1)

plt.figure(figsize=(6, 4))
cdf_groups = {
    'Local-First': baseline_income_df[baseline_income_df['algorithm'] == 'Local-First'].sort_values('driver_index')['income'].astype(float).tolist(),
    'Demand-Greedy': baseline_income_df[baseline_income_df['algorithm'] == 'Demand-Greedy'].sort_values('driver_index')['income'].astype(float).tolist(),
    'PPO (alpha=0.2)': ppo_income_df.sort_values('driver_index')['income'].astype(float).tolist(),
}
for label, values in cdf_groups.items():
    sorted_values = sorted(values)
    y_values = [(idx + 1) / len(sorted_values) for idx in range(len(sorted_values))]
    plt.step(sorted_values, y_values, where='post', label=label)
plt.xlabel('Driver income')
plt.ylabel('CDF')
plt.title('Driver income CDF in the shock scene')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shock_income_cdf.png', dpi=150)
plt.close()
english_bar.update(1)

ppo_shock_df = summary_df[(summary_df['algorithm'] == 'PPO') & (summary_df['scene'] == 'shock')].sort_values('alpha')
plt.figure(figsize=(6, 4))
for _, row in ppo_shock_df.iterrows():
    plt.errorbar(
        row['gini_mean'],
        row['revenue_mean'],
        xerr=row['gini_sem'],
        yerr=row['revenue_sem'],
        fmt='o',
        capsize=5,
        label=f"alpha={row['alpha']:.1f}",
    )
plt.xlabel('Final episode Gini')
plt.ylabel('Episode revenue')
plt.title('Alpha trade-off in the shock scene')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alpha_tradeoff.png', dpi=150)
plt.close()
english_bar.update(1)

english_bar.close()
set_status('English figures finished. The figures/ folder now contains the final English plots.')


In [ ]:
set_status('Packaging refreshed submission bundle...')

package_bar = tqdm(total=3, desc='Packaging', unit='step')
archive_base = '/content/submission_ready_final'
archive_path = archive_base + '.zip'

if os.path.exists(archive_path):
    os.remove(archive_path)
package_bar.update(1)

shutil.make_archive(archive_base, 'zip', '/content', 'submission_ready')
package_bar.update(1)

files.download(archive_path)
package_bar.update(1)
package_bar.close()

set_status('All done. The refreshed submission_ready zip is being downloaded.')
